# 02 — Couche silver : nettoyage et typage

bronze/enedis_conso ──règles de nettoyage──► silver/enedis_conso

bronze/meteo_departements + ref ──► silver/meteo_regions

Chaque règle est nommée et comptée : on doit pouvoir justifier chaque ligne retirée.

In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, format_tables, LAKE

spark = creer_session("silver")
FMT = format_tables(spark)

conso = spark.read.format(FMT).load(f"{LAKE}/bronze/enedis_conso")
meteo = spark.read.format(FMT).load(f"{LAKE}/bronze/meteo_departements")
ref   = spark.read.format(FMT).load(f"{LAKE}/bronze/ref_departements")
print(f"bronze enedis : {conso.count():,} lignes")
print(f"bronze météo  : {meteo.count():,} lignes")

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
bronze enedis : 17,239,680 lignes
bronze météo  : 309,696 lignes


In [2]:
for col in ["profil", "plage_de_puissance_souscrite", "region"]:
    conso.groupBy(col).count().orderBy(col).show(30, truncate=False)

+----------------------+-------+
|profil                |count  |
+----------------------+-------+
|ENT1 (+ ENT2)         |210240 |
|ENT3 (+ ENT4 + ENT5)  |210240 |
|PRO1 (+ PRO1WE)       |2102400|
|PRO2 (+ PRO2WE + PRO6)|1892160|
|PRO3                  |1471680|
|PRO4                  |1051200|
|PRO5                  |2102400|
|RES1 (+ RES1WE)       |630720 |
|RES11 (+ RES11WE)     |1681920|
|RES2 (+ RES5)         |1892160|
|RES2WE                |1261440|
|RES3                  |1471680|
|RES4                  |1261440|
+----------------------+-------+

+----------------------------+-------+
|plage_de_puissance_souscrite|count  |
+----------------------------+-------+
|P0: Total <= 36 kVA         |2733120|
|P1 : ]0-12] kVA             |210240 |
|P1: ]0-3] kVA               |630720 |
|P1: ]0-6] kVA               |630720 |
|P1: ]0-9] kVA               |630720 |
|P2: ]3-6] kVA               |630720 |
|P3: ]6-9] kVA               |1261440|
|P4: ]9-12] kVA              |1892160|
|P5: ]12-

In [3]:
E = "total_energie_soutiree_wh_"
energie = F.expr(f"try_cast({E} AS DOUBLE)")
points  = F.expr("try_cast(nb_points_soutirage AS INT)")

conso.select(
    F.sum((F.col(E) == "S").cast("int")).alias("energie_S"),
    F.sum(F.col(E).isNull().cast("int")).alias("energie_vide"),
    F.sum((energie < 0).cast("int")).alias("energie_negative"),
    F.sum((points <= 0).cast("int")).alias("points_nuls_ou_negatifs"),
    F.sum(F.col("plage_de_puissance_souscrite").startswith("P0").cast("int")).alias("lignes_P0_total"),
).show(vertical=True)

-RECORD 0--------------------------
 energie_S               | 1538158 
 energie_vide            | 0       
 energie_negative        | 0       
 points_nuls_ou_negatifs | 0       
 lignes_P0_total         | 2733120 



In [4]:
conso.select(F.min("horodate"), F.max("horodate")).show(truncate=False)
print("demi-heures différentes :", conso.select("horodate").distinct().count())
print("attendu sur une année   :", 365 * 48)

+-------------------+-------------------+
|min(horodate)      |max(horodate)      |
+-------------------+-------------------+
|2021-12-31 23:00:00|2022-12-31 22:30:00|
+-------------------+-------------------+

demi-heures différentes : 17520
attendu sur une année   : 17520


In [5]:
meteo.select(F.min("date"), F.max("date")).show()
print("lignes météo hors 2022 :", meteo.filter(~F.col("date").startswith("2022")).count())

sans_region = meteo.join(ref, meteo.code_insee_departement == ref.code_departement, "left_anti")
print("lignes météo sans région :", sans_region.count())
sans_region.select("code_insee_departement", "departement").distinct().show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2018-01-01|2026-10-31|
+----------+----------+

lignes météo hors 2022 : 274656
lignes météo sans région : 0
+----------------------+-----------+
|code_insee_departement|departement|
+----------------------+-----------+
+----------------------+-----------+



In [6]:
p0 = conso.filter(F.col("plage_de_puissance_souscrite").startswith("P0"))
print("lignes P0                   :", p0.count())
print("lignes P0 avec énergie = S  :", p0.filter(F.col(E) == "S").count())

cles = ["horodate", "code_region", "profil", "plage_de_puissance_souscrite"]
print("doublons (toutes plages)    :", conso.count() - conso.dropDuplicates(cles).count())

lignes P0                   : 2733120
lignes P0 avec énergie = S  : 157680
doublons (toutes plages)    : 0


## 1. Règles de nettoyage

Chaque règle est nommée et comptée sur le bronze. Une ligne peut enfreindre plusieurs règles.

In [7]:
from functools import reduce

spark.conf.set("spark.sql.session.timeZone", "UTC")   # les horodates Enedis sont en heure UTC
heure_paris = F.from_utc_timestamp(F.to_timestamp("horodate"), "Europe/Paris")

regles = {
    "plage_detaillee_hors_P0":    ~F.col("plage_de_puissance_souscrite").startswith("P0"),
    "energie_secret_statistique": F.col(E) == "S",
    "energie_vide_ou_negative":   F.col(E).isNull() | (energie < 0),
    "aucun_point_de_soutirage":   points.isNull() | (points <= 0),
    "hors_2022_heure_de_paris":   F.year(heure_paris) != 2022,
}
conso.select([F.sum(cond.cast("int")).alias(nom) for nom, cond in regles.items()]).show(vertical=True)

-RECORD 0------------------------------
 plage_detaillee_hors_P0    | 14506560 
 energie_secret_statistique | 1538158  
 energie_vide_ou_negative   | 0        
 aucun_point_de_soutirage   | 0        
 hors_2022_heure_de_paris   | 0        



In [8]:
a_exclure = reduce(lambda x, y: x | y, regles.values())
garde = conso.filter(~a_exclure)

n_bronze, n_garde = conso.count(), garde.count()
print(f"bronze  : {n_bronze:,} lignes")
print(f"gardées : {n_garde:,} lignes ({n_garde / n_bronze:.1%} du bronze)")
print(f"retirées: {n_bronze - n_garde:,} lignes")

bronze  : 17,239,680 lignes
gardées : 2,575,440 lignes (14.9% du bronze)
retirées: 14,664,240 lignes


## 2. Silver de la consommation : typage et enrichissement

In [9]:
noms_regions = ref.select("code_region", "nom_region").distinct()

silver_conso = (garde
    .withColumn("horodate", heure_paris)                          # heure de Paris (découverte 2)
    .withColumn("date", F.to_date("horodate"))
    .withColumn("heure", F.hour("horodate"))
    .withColumn("energie_wh", energie)                            # texte → nombre
    .withColumn("nb_points", points)                              # texte → nombre entier
    .withColumn("puissance_mw", F.col("energie_wh") * 2 / 1e6)    # Wh par 1/2 h → MW moyens
    .withColumn("famille_profil", F.substring("profil", 1, 3))    # RES, PRO ou ENT
    .join(noms_regions, "code_region", "left")                    # nom de région correct (découverte 1)
    .select("horodate", "date", "heure", "code_region", "nom_region",
            "profil", "famille_profil", "nb_points", "energie_wh", "puissance_mw"))

silver_conso.printSchema()
silver_conso.show(5, truncate=False)

root
 |-- horodate: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- heure: integer (nullable = true)
 |-- code_region: string (nullable = true)
 |-- nom_region: string (nullable = true)
 |-- profil: string (nullable = true)
 |-- famille_profil: string (nullable = true)
 |-- nb_points: integer (nullable = true)
 |-- energie_wh: double (nullable = true)
 |-- puissance_mw: double (nullable = true)

+-------------------+----------+-----+-----------+------------------+--------------------+--------------+---------+------------+------------+
|horodate           |date      |heure|code_region|nom_region        |profil              |famille_profil|nb_points|energie_wh  |puissance_mw|
+-------------------+----------+-----+-----------+------------------+--------------------+--------------+---------+------------+------------+
|2022-12-09 03:30:00|2022-12-09|3    |28         |Normandie         |RES2 (+ RES5)       |RES           |865927   |5.94060665E8|1188.12133  |
|2022-12-09 03

In [10]:
silver_conso.write.format(FMT).mode("overwrite").save(f"{LAKE}/silver/enedis_conso")

relu = spark.read.format(FMT).load(f"{LAKE}/silver/enedis_conso")
print(f"silver/enedis_conso : {relu.count():,} lignes ({relu.count() / n_bronze:.1%} du bronze)")
print("régions sans nom :", relu.filter(F.col("nom_region").isNull()).count())
relu.groupBy("nom_region").count().orderBy("nom_region").show(truncate=False)
relu.groupBy("famille_profil").count().show()

silver/enedis_conso : 2,575,440 lignes (14.9% du bronze)
régions sans nom : 0
+--------------------------+------+
|nom_region                |count |
+--------------------------+------+
|Auvergne-Rhône-Alpes      |227760|
|Bourgogne-Franche-Comté   |210240|
|Bretagne                  |210240|
|Centre-Val de Loire       |210240|
|Grand Est                 |227760|
|Hauts-de-France           |210240|
|Normandie                 |210240|
|Nouvelle-Aquitaine        |210240|
|Occitanie                 |210240|
|Pays de la Loire          |210240|
|Provence-Alpes-Côte d'Azur|210240|
|Île-de-France             |227760|
+--------------------------+------+

+--------------+-------+
|famille_profil|  count|
+--------------+-------+
|           ENT| 262800|
|           RES|1261440|
|           PRO|1051200|
+--------------+-------+



## 3. Silver de la météo : 2022, par région et par jour

In [11]:
meteo_2022 = (meteo
    .withColumn("date", F.to_date("date"))
    .filter(F.year("date") == 2022)
    .withColumn("tmin", F.expr("try_cast(tmin_c_ AS DOUBLE)"))
    .withColumn("tmax", F.expr("try_cast(tmax_c_ AS DOUBLE)"))
    .withColumn("tmoy", F.expr("try_cast(tmoy_c_ AS DOUBLE)"))
    .join(ref.select("code_departement", "code_region", "nom_region"),
          F.col("code_insee_departement") == F.col("code_departement"), "inner"))

print(f"météo 2022 : {meteo_2022.count():,} lignes (attendu 96 × 365 = 35,040)")
print("températures moyennes vides :", meteo_2022.filter(F.col("tmoy").isNull()).count())

météo 2022 : 35,040 lignes (attendu 96 × 365 = 35,040)
températures moyennes vides : 0


In [12]:
silver_meteo = (meteo_2022
    .groupBy("date", "code_region", "nom_region")
    .agg(F.round(F.avg("tmoy"), 2).alias("tmoy_c"),
         F.round(F.avg("tmin"), 2).alias("tmin_c"),
         F.round(F.avg("tmax"), 2).alias("tmax_c"),
         F.count("*").alias("nb_departements")))

silver_meteo.write.format(FMT).mode("overwrite").save(f"{LAKE}/silver/meteo_regions")

relu_m = spark.read.format(FMT).load(f"{LAKE}/silver/meteo_regions")
print(f"silver/meteo_regions : {relu_m.count():,} lignes")
relu_m.groupBy("nom_region").agg(
    F.count("*").alias("jours"),
    F.min("tmoy_c").alias("jour_le_plus_froid"),
    F.max("tmoy_c").alias("jour_le_plus_chaud")
).orderBy("nom_region").show(truncate=False)

silver/meteo_regions : 4,745 lignes
+--------------------------+-----+------------------+------------------+
|nom_region                |jours|jour_le_plus_froid|jour_le_plus_chaud|
+--------------------------+-----+------------------+------------------+
|Auvergne-Rhône-Alpes      |365  |-1.82             |28.05             |
|Bourgogne-Franche-Comté   |365  |-4.03             |28.39             |
|Bretagne                  |365  |1.42              |28.42             |
|Centre-Val de Loire       |365  |-2.99             |27.8              |
|Corse                     |365  |7.46              |28.67             |
|Grand Est                 |365  |-6.2              |26.85             |
|Hauts-de-France           |365  |-3.59             |30.11             |
|Normandie                 |365  |-2.23             |28.33             |
|Nouvelle-Aquitaine        |365  |0.1               |29.53             |
|Occitanie                 |365  |0.31              |28.88             |
|Pays de la Loi

## 4. Bilan du silver

| Table | Lignes bronze | Lignes silver | Ce qu'on a fait |
|---|---|---|---|
| `silver/enedis_conso` | 17 239 680 | 2 575 440 | Lignes P0 seulement (pas de double comptage), « S » et valeurs vides retirés, heure UTC → heure de Paris, région nommée via `ref`, puissance en MW |
| `silver/meteo_regions` | 309 696 | 4 745 | Année 2022 seulement, départements rattachés à leur région, moyenne par région et par jour |

**Limites à retenir :**
- 9 séries de consommation sont entièrement masquées par le secret statistique (« S ») : certaines régions ont 12 séries de profils, d'autres 13.
- La moyenne régionale de température donne le même poids à chaque département, quelle que soit sa population.
- La Corse est présente dans la météo mais pas chez Enedis : elle disparaîtra lors du croisement dans le gold.